# YOLO data-pipeline ablations

Two independent 30-epoch experiments on the same grouped fold: COCO polygon rasterization only, then disk normalization only. Baseline architecture, mask_ratio2, annotations and augmentation are preserved. Each produces validation.csv, saved masks and submission.csv. No automatic submission.


In [ ]:
import os,sys,subprocess,json
from pathlib import Path
repo=Path('/kaggle/working/solar-fil-yolo-data')
subprocess.run(['git','clone','https://github.com/Dharun235/kaggle-solar-filament-segmentation.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','ultralytics==8.4.152','pycocotools'],check=True)
import torch
assert torch.cuda.is_available(), 'Enable GPU'
root=Path('/kaggle/working/yolo_data_ablation')
root.mkdir(exist_ok=True)
(root/'git_sha.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],text=True))
(root/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))


In [ ]:
results=[]
for variant in ['coco','disk']:
    run=root/variant
    (root/'experiment_status.json').write_text(json.dumps({'current':variant,'completed':results},indent=2))
    subprocess.run([sys.executable,'models/yolo_instance.py','--data-root','/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026','--run-dir',str(run),'--data-pipeline',variant],check=True)
    selected=json.loads((run/'selected.json').read_text())
    results.append({'variant':variant,**selected})
    (root/'comparison.json').write_text(json.dumps(results,indent=2))
    print('COMPLETED',variant,selected,flush=True)
(root/'experiment_status.json').write_text(json.dumps({'current':'complete','completed':results},indent=2))


In [ ]:
print(json.dumps(results,indent=2))
print('Compare with baseline PQ 0.4012144075536765 and the periodic-checkpoint baseline when complete.')
